# B3Pdb and CPPsite2 peptide dataset acquisition and preprocessing

This notebook retrieves and prepares peptide sequence data used for fine-tuning the peptide generator. The aim is to build a clean peptide corpus from public peptide resources and convert it into a format suitable for tokenisation, language-model training, and downstream peptide generation.

The notebook uses two public peptide resources:

1. **B3Pdb chemical-modification peptide entries**  
   URL: `https://webs.iiitd.edu.in/raghava/b3pdb/browse_sub1.php?token=Chemical&col=10`  
   Rationale: B3Pdb provides blood-brain barrier peptide entries. The chemical-modification subset is used here as a BBB-relevant peptide source for the fine-tuning corpus.

2. **CPPsite2 natural peptide FASTA file**  
   URL: `http://crdd.osdd.net/raghava/cppsite/downloads.php`  
   Rationale: CPPsite2 provides cell-penetrating peptide sequences. These peptides are not treated as BBB-positive labels, but they enrich the peptide language-model corpus with uptake-related peptide patterns.

The notebook performs the following steps:

1. define local raw and processed data paths,
2. retrieve public B3Pdb peptide data from available online resources,
3. parse downloaded files or HTML tables when needed,
4. read CPPsite2 peptide FASTA sequences,
5. standardise peptide sequences to canonical one-letter amino-acid format,
6. remove invalid, duplicated, or out-of-scope sequences,
7. save cleaned peptide tables for downstream model training.

Important scope note:

This notebook is a **data acquisition and cleaning step**, not a biological validation analysis. The resulting peptide corpus is used as a training/fine-tuning resource for sequence generation. Peptide function, receptor specificity, toxicity, synthesizability, and BBB transport relevance are not assumed from this dataset alone.

In the wider prototype, this notebook provides the peptide-data preparation stage:

**public peptide data retrieval → sequence cleaning → peptide corpus construction → transformer fine-tuning → peptide generation**

## 1. Retrieve B3Pdb chemical-modification peptide pages

The B3Pdb chemical-modification browse pages are retrieved from the public B3Pdb web interface. Each page is parsed as an HTML table, the largest table is retained, and all pages are concatenated into one raw table. The raw table is saved before sequence cleaning so the acquisition step remains reproducible.

In [ ]:
# -------------------------
# 1.1 Retrieve all B3Pdb chemical-modification pages
# -------------------------

from io import StringIO
from pathlib import Path
import pandas as pd
import requests

raw_dir = Path("../data/raw/peptides/b3pdb")
raw_dir.mkdir(parents=True, exist_ok=True)

base_url = "https://webs.iiitd.edu.in/raghava/b3pdb/browse_sub1.php?token=Chemical&col=10"

all_pages = []

for page in range(1, 12):
    url = base_url if page == 1 else f"{base_url}&page={page}"
    print("Reading:", url)

    html = requests.get(url, timeout=20).text
    tables = pd.read_html(StringIO(html))

    table = max(tables, key=lambda x: x.shape[0] * x.shape[1])
    table["source_page"] = page
    table["source_url"] = url

    all_pages.append(table)

raw_b3pdb_chemical = (
    pd.concat(all_pages, ignore_index=True)
    .drop_duplicates()
    .reset_index(drop=True)
)

print("Raw B3Pdb chemical table shape:", raw_b3pdb_chemical.shape)
display(raw_b3pdb_chemical.head())

In [ ]:
# -------------------------
# 1.2 Save raw B3Pdb table
# -------------------------

raw_b3pdb_path = raw_dir / "b3pdb_chemical_raw.csv"

raw_b3pdb_chemical.to_csv(
    raw_b3pdb_path,
    index=False,
)

print("Saved raw B3Pdb chemical table:", raw_b3pdb_path)

## 2. Clean B3Pdb peptide sequences

The raw B3Pdb table contains peptide sequences in one-letter amino-acid format. These sequences are standardised by converting to uppercase, removing non-letter characters, retaining only canonical amino acids, dropping missing values, and removing duplicates. The cleaned B3Pdb sequence table is saved for downstream model training.

In [ ]:
# -------------------------
# 2.1 Define peptide-cleaning helper
# -------------------------

import re

processed_dir = Path("../data/processed/peptides")
processed_dir.mkdir(parents=True, exist_ok=True)

canonical_aas = set("ACDEFGHIKLMNPQRSTVWY")


def clean_peptide(seq):
    if seq is None or pd.isna(seq):
        return None

    seq = str(seq).strip().upper()
    seq = re.sub(r"[^A-Z]", "", seq)

    if seq in {"", "NA", "NAN", "NONE", "NULL"}:
        return None

    if not set(seq).issubset(canonical_aas):
        return None

    return seq

In [ ]:
# -------------------------
# 2.2 Extract clean B3Pdb sequences
# -------------------------

seq_col = "PEPTIDE SEQUENCE (1-letter)"

b3pdb_chemical_sequences = (
    raw_b3pdb_chemical[seq_col]
    .apply(clean_peptide)
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
    .to_frame("sequence")
)

b3pdb_chemical_sequences["source"] = "B3PDB_Chemical"
b3pdb_chemical_sequences["bbb_positive"] = 1
b3pdb_chemical_sequences["n_amino_acids"] = (
    b3pdb_chemical_sequences["sequence"].str.len()
)

print("Clean unique B3Pdb chemical sequences:", b3pdb_chemical_sequences.shape)
display(b3pdb_chemical_sequences.head(20))

In [ ]:
# -------------------------
# 2.3 Save clean B3Pdb sequences
# -------------------------

b3pdb_sequences_path = processed_dir / "b3pdb_chemical_unique_sequences.csv"

b3pdb_chemical_sequences.to_csv(
    b3pdb_sequences_path,
    index=False,
)

print("Saved clean B3Pdb sequences:", b3pdb_sequences_path)

## 3. Read and clean CPPsite2 natural peptide sequences

A natural CPPsite2 peptide FASTA file is used as an additional peptide corpus. These sequences are uptake-related peptide examples, not BBB-positive labels. They are cleaned using the same canonical amino-acid filter as B3Pdb and retained as a broader peptide pretraining/fine-tuning resource.

In [ ]:
# -------------------------
# 3.1 Define FASTA reader
# -------------------------

def read_fasta_sequences(path):
    sequences = []
    current = []

    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()

            if not line:
                continue

            if line.startswith(">"):
                if current:
                    sequences.append("".join(current))
                    current = []
            else:
                current.append(line)

    if current:
        sequences.append("".join(current))

    return sequences

In [ ]:
# -------------------------
# 3.2 Read and clean CPPsite2 natural peptides
# -------------------------

cpp_path = Path("../data/raw/peptides/cppsite2/natural_pep.fa")

cpp_sequences = (
    pd.Series(read_fasta_sequences(cpp_path), name="sequence")
    .apply(clean_peptide)
    .dropna()
    .drop_duplicates()
    .reset_index(drop=True)
    .to_frame()
)

cpp_sequences["source"] = "CPPsite2_natural"
cpp_sequences["bbb_positive"] = 0  # uptake-positive, not necessarily BBB-positive
cpp_sequences["n_amino_acids"] = cpp_sequences["sequence"].str.len()

print("Clean unique CPPsite2 sequences:", cpp_sequences.shape)
display(cpp_sequences.head(20))

## 4. Combine B3Pdb and CPPsite2 peptide corpora

The cleaned B3Pdb and CPPsite2 sequences are combined into one unique peptide corpus for downstream language-model training. Duplicate sequences are removed. The `source` column records the first retained source label, and `bbb_positive` should be interpreted cautiously because CPPsite2 sequences are uptake-related rather than confirmed BBB-shuttle peptides.

In [ ]:
# -------------------------
# 4.1 Combine B3Pdb and CPPsite2 peptide datasets
# -------------------------

b3pdb_sequences = pd.read_csv(
    processed_dir / "b3pdb_chemical_unique_sequences.csv"
)

combined_peptides = pd.concat(
    [cpp_sequences, b3pdb_sequences],
    ignore_index=True,
)

combined_peptides = (
    combined_peptides
    .drop_duplicates(subset=["sequence"])
    .reset_index(drop=True)
)

print("Combined unique peptides:", combined_peptides.shape)

print("Source counts:")
display(combined_peptides["source"].value_counts().to_frame("n_sequences"))

display(combined_peptides.head(20))

In [ ]:
# -------------------------
# 4.2 Save combined peptide corpus
# -------------------------

combined_peptides_path = processed_dir / "cppsite_b3pdb_training_sequences.csv"

combined_peptides.to_csv(
    combined_peptides_path,
    index=False,
)

print("Saved combined peptide corpus:", combined_peptides_path)

## Final note

This notebook exports the cleaned peptide corpus used by downstream peptide language-model notebooks. The corpus is intended for generative model training and fine-tuning, not as a validated functional peptide benchmark.

Output file:

`../data/processed/peptides/cppsite_b3pdb_training_sequences.csv`